<a href="https://colab.research.google.com/github/amitpaudel21/ARTIFICIAL_INTELLIGENCE_LAB/blob/main/1BF24CS042_WEEK9_ALPHA_BETA_PRUNING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import math

pruned = []
values = {}  # node path -> value found by the search


def label(path):
    """Node name by branches taken from the root, e.g. L-R."""
    return "-".join("LR"[i] if i < 2 else str(i) for i in path)


def max_value(state, alpha, beta, path):
    if not isinstance(state, list):          # terminal test
        return state, [state]
    v, best = -math.inf, []
    for i, child in enumerate(state):
        value, sub = min_value(child, alpha, beta, path + [i])
        if value > v:
            v, best = value, [label(path + [i])] + sub
        if v >= beta:                        # beta cutoff
            record_pruned(state, i, path, alpha, beta)
            values[tuple(path)] = v
            return v, best
        alpha = max(alpha, v)
    values[tuple(path)] = v
    return v, best


def min_value(state, alpha, beta, path):
    if not isinstance(state, list):
        return state, [state]
    v, best = math.inf, []
    for i, child in enumerate(state):
        value, sub = max_value(child, alpha, beta, path + [i])
        if value < v:
            v, best = value, [label(path + [i])] + sub
        if v <= alpha:                       # alpha cutoff
            record_pruned(state, i, path, alpha, beta)
            values[tuple(path)] = v
            return v, best
        beta = min(beta, v)
    values[tuple(path)] = v
    return v, best


def record_pruned(state, i, path, alpha, beta):
    for j in range(i + 1, len(state)):
        pruned.append((label(path + [j]), state[j], alpha, beta))


def alpha_beta_search(tree):
    pruned.clear()
    values.clear()
    value, path = max_value(tree, -math.inf, math.inf, [])
    return value, ["Root"] + path


def draw_tree(node, path=(), cut=False):
    """Return (lines, width, centre column) for the subtree at this node."""
    if not isinstance(node, list):
        text = f"{node}x" if cut else str(node)
        width = len(text) + 2
        return [text.center(width)], width, width // 2

    kind = "MAX" if len(path) % 2 == 0 else "MIN"
    text = f"{kind}:x" if cut else f"{kind}:{values[path]}"

    cut_names = {entry[0] for entry in pruned}
    kids = [draw_tree(child, path + (i,), cut or label(list(path) + [i]) in cut_names)
            for i, child in enumerate(node)]

    # place the children side by side
    height = max(len(lines) for lines, _, _ in kids)
    body = [""] * height
    centres, offset = [], 0
    for lines, width, centre in kids:
        for row in range(height):
            body[row] += lines[row] if row < len(lines) else " " * width
        centres.append(offset + centre)
        offset += width

    width = max(offset, len(text) + 2)
    mid = (centres[0] + centres[-1]) // 2

    # connector lines between this node and its children
    bar = [" "] * width
    for col in range(centres[0], centres[-1] + 1):
        bar[col] = "-"
    for col in centres:
        bar[col] = "+"
    bar[mid] = "+"

    start = max(0, min(width - len(text), mid - len(text) // 2))
    top = (" " * start + text).ljust(width)
    return [top, "".join(bar)] + [line.ljust(width) for line in body], width, mid


def print_tree(tree):
    lines, _, _ = draw_tree(tree)
    print("\n".join(line.rstrip() for line in lines))
    print("\n(x = pruned, never explored)")


if __name__ == "__main__":
    # MAX -> MIN -> MAX -> leaves (tree from the problem slide)
    tree = [
        [[10, 9], [14, 18]],
        [[5, 4], [50, 3]],
    ]

    value, path = alpha_beta_search(tree)

    print("Game tree:\n")
    print_tree(tree)
    print()
    print("Value of root node:", value)
    print("Optimal path:", "  =>  ".join(map(str, path)))
    print("\nPruned branches:")
    for name, subtree, alpha, beta in pruned:
        print(f"  {name}  {subtree}  (alpha={alpha}, beta={beta})")

Game tree:

            MAX:10
       +-------+-------+
    MIN:10           MIN:5
   +---+----+      +---+----+
MAX:10   MAX:14  MAX:5    MAX:x
  ++-+    +-+-+   ++-+    +-+--+
 10  9   14  18x  5  4   50x  3x

(x = pruned, never explored)

Value of root node: 10
Optimal path: Root  =>  L  =>  L-L  =>  L-L-L  =>  10

Pruned branches:
  L-R-R  18  (alpha=-inf, beta=10)
  R-R  [50, 3]  (alpha=10, beta=inf)
